# SMS Spam Classification: A Complete Machine Learning & Deep Learning Analysis

| **Project Info** | **Details** |
| :--- | :--- |
| **Author** | Shabiha |
| **Domain** | Natural Language Processing & Deep Learning |
| **Objective** | Build an interpretative NLP model pipeline to automatically classify SMS messages as Spam or Ham. |

## Problem Statement
In the modern digital landscape, the proliferation of unsolicited and often malicious spam messages is a significant nuisance and security threat. This project aims to build a robust Natural Language Processing (NLP) pipeline to automatically classify SMS messages as either **Spam** or **Ham** (legitimate). 

## Analytical Objectives
1. **Data Preprocessing & EDA**: Clean the dataset and perform exploratory data analysis to identify key textual patterns.
2. **Traditional Machine Learning Models**: Establish strong baselines using multiple classifiers (Naive Bayes, Decision Trees, SVM, Gradient Boosting, etc.).
3. **Deep Learning (Bi-LSTM)**: Implement an advanced Bidirectional LSTM neural network using TensorFlow/Keras to capture temporal dependencies in text and optimize performance.
4. **Business Impact**: Deliver reliable, interpretative models to minimize false positives, ensuring an excellent user experience.

## Environment Setup
Run the following cell to install all required dependencies (if you haven't already).

In [ ]:
%pip install numpy pandas matplotlib seaborn nltk wordcloud scikit-learn tensorflow ipython

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns
import nltk, re, collections, pickle, os
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize
from wordcloud import WordCloud

from IPython.display import display

from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB, MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import GradientBoostingClassifier, BaggingClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import confusion_matrix, classification_report

import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.layers import Dense, Embedding, LSTM, Dropout, Bidirectional
from tensorflow.keras.models import Sequential
from tensorflow.keras.optimizers import Adam

import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (15, 5)
plt.style.use('ggplot')
seed = 42

nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('punkt')
nltk.download('punkt_tab')

## 1. Helper Functions for Visualization

In [ ]:
def plot_history(history):
    loss_list = [s for s in history.history.keys() if 'loss' in s and 'val' not in s]
    val_loss_list = [s for s in history.history.keys() if 'loss' in s and 'val' in s]
    acc_list = [s for s in history.history.keys() if 'accuracy' in s and 'val' not in s]
    val_acc_list = [s for s in history.history.keys() if 'accuracy' in s and 'val' in s]

    plt.figure(figsize = (12, 5), dpi = 100)
    epochs = range(1, len(history.history[loss_list[0]]) + 1)

    plt.subplot(1, 2, 1)
    for l in loss_list: plt.plot(epochs, history.history[l], 'b-o', label=f'Train ({history.history[l][-1]:.4f})')
    for l in val_loss_list: plt.plot(epochs, history.history[l], 'g', label=f'Valid ({history.history[l][-1]:.4f})')
    plt.title('Loss Dynamics')
    plt.xlabel('Epochs')
    plt.legend(loc='best')
    plt.grid(True)

    plt.subplot(1, 2, 2)
    for l in acc_list: plt.plot(epochs, history.history[l], 'b-o', label=f'Train ({history.history[l][-1]:.4f})')
    for l in val_acc_list: plt.plot(epochs, history.history[l], 'g', label=f'Valid ({history.history[l][-1]:.4f})')
    plt.title('Accuracy Dynamics')
    plt.xlabel('Epochs')
    plt.legend(loc='best')
    plt.grid(True)
    plt.tight_layout()
    plt.show()

def plot_conf_matr(conf_matr, classes, normalize=False, title='Confusion matrix', cmap=plt.cm.winter):
    import itertools
    accuracy = np.trace(conf_matr) / np.sum(conf_matr).astype('float')
    plt.figure(figsize=(8, 6))
    plt.imshow(conf_matr, interpolation='nearest', cmap=cmap)
    plt.title(f'\n{title}\n', fontsize=16)
    plt.colorbar()
    tick_marks = np.arange(len(classes))
    plt.xticks(tick_marks, classes, rotation=45)
    plt.yticks(tick_marks, classes)

    thresh = conf_matr.max() / 2
    for i, j in itertools.product(range(conf_matr.shape[0]), range(conf_matr.shape[1])):
        plt.text(j, i, f"{conf_matr[i, j]:,}",
                 horizontalalignment="center",
                 fontweight='bold',
                 color="white" if conf_matr[i, j] > thresh else "black")
    plt.tight_layout()
    plt.ylabel('True label')
    plt.xlabel(f'Predicted label\n\nAccuracy = {accuracy*100:0.2f}%; Error = {(1-accuracy)*100:0.2f}%')
    plt.show()

def word_cloud(tag, df):
    df_words_nl = ' '.join(list(df[df['feature'] == tag]['message']))
    df_wc_nl = WordCloud(width = 600, height = 512).generate(df_words_nl)
    plt.figure(figsize = (10, 7), facecolor = 'k')
    plt.imshow(df_wc_nl)
    plt.axis('off')
    plt.title(f'Word Cloud for {tag.upper()}', color='white', fontsize=18)
    plt.tight_layout(pad = 1)
    plt.show()

## 2. Dataset Loading & Cleaning

In [ ]:
file_path = r'C:\Users\Sanman\Downloads\Projects\SMS_Spam_Classification_ML_DL\Data\spam.csv'
df_spam = pd.read_csv(file_path, encoding='latin-1')

df_spam = df_spam.filter(['v1', 'v2'], axis=1)
df_spam.columns = ['feature', 'message']
df_spam.drop_duplicates(inplace=True, ignore_index=True)
print(f'Dataset Shape: {df_spam.shape}')
display(df_spam.head())

## 3. Exploratory Data Analysis (EDA)
Understanding class balance and most prevalent vocabulary.

In [ ]:
plt.figure(figsize=(8, 5))
ax = sns.countplot(x=df_spam['feature'])
plt.title('Distribution of Spam vs Ham')
for p in ax.patches:
    ax.annotate(f'{100 * p.get_height() / len(df_spam):.2f}%', (p.get_x() + p.get_width() / 2., p.get_height()), ha='center', va='bottom')
plt.show()

In [ ]:
word_cloud('spam', df_spam)
word_cloud('ham', df_spam)

### EDA Insights
- **Class Imbalance**: The dataset is highly imbalanced with 'Ham' messages dominating (approx 87%). Accuracy alone could be a misleading metric. We must focus on avoiding False Positives.
- **Spam Vocabulary**: Spam messages heavily rely on promotional, urgent, or incentive-related terms (e.g., 'free', 'win', 'prize', 'call').
- **Ham Vocabulary**: Ham messages showcase conversational and everyday language (e.g., 'ok', 'go', 'come', 'time').

## 4. Traditional Machine Learning Models
Data transformations include basic regex for URLs, money, phone numbers, and TF-IDF feature extraction via WordNet Lemmatization.

In [ ]:
size_vocabulary = 1000
lemmatizer = WordNetLemmatizer()
full_df_l = []

for i in range(df_spam.shape[0]):
    mess_1 = df_spam.iloc[i]['message']
    mess_1 = re.sub(r'\b[\w\-.]+?@\w+?\.\w{2,4}\b', 'emailaddr', mess_1)
    mess_1 = re.sub(r'(http[s]?\S+)|(\w+\.[A-Za-z]{2,4}\S*)', 'httpaddr', mess_1)
    mess_1 = re.sub(r'£|\$', 'moneysymb', mess_1)
    mess_1 = re.sub(r'\b(\+\d{1,2}\s)?\d?[\-(.]?\d{3}\)?[\s.-]?\d{3}[\s.-]?\d{4}\b', 'phonenumbr', mess_1)
    mess_1 = re.sub(r'\d+(\.\d+)?', 'numbr', mess_1)
    mess_1 = re.sub(r'[^\w\d\s]', ' ', mess_1)
    mess_1 = re.sub(r'[^A-Za-z]', ' ', mess_1).lower()
    
    token_messages = word_tokenize(mess_1)
    mess = [lemmatizer.lemmatize(word) for word in token_messages if word not in set(stopwords.words('english'))]
    full_df_l.append(" ".join(mess))

vectorizer = CountVectorizer(max_features=size_vocabulary)
X = vectorizer.fit_transform(full_df_l).toarray()
y = df_spam['feature'].map({'ham': 1, 'spam': 0}).values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=seed)

In [ ]:
models = {
    'Gaussian Naive Bayes': GaussianNB(),
    'Multinomial Naive Bayes': MultinomialNB(),
    'Decision Tree': DecisionTreeClassifier(random_state=seed),
    'Logistic Regression': LogisticRegression(random_state=seed, solver='liblinear'),
    'K-Nearest Neighbors': KNeighborsClassifier(n_neighbors=3),
    'Support Vector Machine': SVC(probability=True, random_state=seed),
    'Gradient Boosting': GradientBoostingClassifier(random_state=seed),
    'Bagging Classifier': BaggingClassifier(random_state=seed)
}

for name, clf in models.items():
    print(f"\n{'='*40}\nEvaluating: {name}\n{'='*40}")
    clf.fit(X_train, y_train)
    y_pred = clf.predict(X_test)
    print("\nClassification Report:\n", classification_report(y_test, y_pred, target_names=['Spam', 'Ham']))
    conf_m = confusion_matrix(y_test, y_pred)
    plot_conf_matr(conf_m, classes=['Spam', 'Ham'], title=f'{name}')

### Performance Summary of Traditional Models

| Classifier                | Accuracy | Error Rate | Key Interpretation                                                                                   |
| ------------------------- | -------- | ---------- | ---------------------------------------------------------------------------------------------------- |
| Logistic Regression / SVC | ~97.68%   | ~2.32%      | Best in Class: Provides the highest overall accuracy and near-perfect protection of legitimate mail. |
| Gradient Boosting         | ~97.45%   | ~2.55%      | High Precision: Excellent at ensuring flagged spam is truly junk, though slightly lower spam recall. |
| Multinomial Naive Bayes   | ~97.29%   | ~2.71%      | Balanced: Best middle-ground model with high consistency across both spam and ham classes.           |
| Bagging Classifier        | ~96.21%   | ~3.79%      | Reliable: Strong ham protection but allows about 15% of spam to pass.                                |
| Decision Tree             | ~95.82%   | ~4.18%      | Solid All-rounder: High performance, but slightly more misclassifications than ensemble methods.     |
| K-Nearest Neighbors       | ~94.43%   | ~5.57%      | Moderate: High ham recall, but struggles with spam detection (≈31% missed).                          |
| Gaussian Naive Bayes      | ~79.43%   | ~20.57%     | Poor: Trigger-happy model that flags many legitimate messages as spam.                               |

## 5. Advanced Deep Learning approach (Bi-LSTM)
Using sequence processing and word embeddings, a Bidirectional LSTM captures semantic relationships and patterns missed by traditional Bag-of-Words approaches.

In [ ]:
embedding_dimension = 64
tokenizer = Tokenizer(num_words=size_vocabulary, oov_token="<OOV>", lower=False)
tokenizer.fit_on_texts(full_df_l)
word_index = tokenizer.word_index
size_voc = len(word_index) + 1

sequences = tokenizer.texts_to_sequences(full_df_l)
max_len = max([len(i) for i in sequences])
padded_set = pad_sequences(sequences, padding='post', maxlen=max_len, truncating='post')

X_train_dl, X_test_dl, y_train_dl, y_test_dl = train_test_split(padded_set, y, test_size=0.25, random_state=seed)

In [ ]:
model = Sequential([
    Embedding(size_voc, embedding_dimension, input_length=max_len),
    Bidirectional(LSTM(100)),
    Dropout(0.3),
    Dense(20, activation='relu'),
    Dropout(0.3),
    Dense(1, activation='sigmoid')
])

model.compile(loss='binary_crossentropy', optimizer=Adam(learning_rate=0.001), metrics=['accuracy'])
model.summary()

In [ ]:
history = model.fit(X_train_dl, y_train_dl, epochs=10, validation_split=0.2, verbose=1)

In [ ]:
plot_history(history)

In [ ]:
y_pred_probs = model.predict(X_test_dl)
y_pred_bLSTM = (y_pred_probs > 0.5).astype(int).flatten()

print("\n--- Bi-LSTM Classification Report ---\n")
print(classification_report(y_test_dl, y_pred_bLSTM, target_names=['Spam', 'Ham']))
plot_conf_matr(confusion_matrix(y_test_dl, y_pred_bLSTM), classes=['Spam', 'Ham'], title='Bi-LSTM Performance')

## 6. Saving Model & Real-world Testing
Finally, we serialize the models and perform a live inference test on incoming text strings.

In [ ]:
M_name = "Spam_Detector_Model"
with open(M_name + ".pkl", "wb") as f:
    pickle.dump(tokenizer, f)
    
model.save(M_name + '.h5')
print(f"Model saved successfully. Size: {os.stat(M_name + '.h5').st_size / 1024:.2f} KB")

# Sample validation
message_example = ["Darling, please give me a cup of tea"]
message_example_tp = pad_sequences(tokenizer.texts_to_sequences(message_example), maxlen=max_len, padding='post', truncating='post')

pred = float(model.predict(message_example_tp, verbose=0)[0][0])
print(f"\nMessage: '{message_example[0]}'")
print("Prediction:", "Ham (Real text)" if pred > 0.5 else "Spam", f"(Confidence: {pred:.4f})")

## Conclusion & Business Utility
This pipeline proves that Deep Learning (**Bi-LSTM**) combined with **Word Embeddings** effectively captures the semantic context of textual payloads, surpassing basic TF-IDF methods like Naive Bayes in nuanced edge cases. 

**Next Steps:**
- Tune probability thresholds on production endpoints to optimize the precision-recall trade-off (prioritizing 0% false positive rates on non-spam emails).
- Deploy model artifact `.h5` via a REST API (e.g. FastAPI/Flask) for instantaneous streaming inference.